### CMPE 49T — HW 1: Part II
 Summary
 
 In this part of the homework, a convolutional neural network (CNN) was implemented
 manually using only NumPy operations. Each layer, including convolution, padding,
 pooling, batch normalization, and activation, was built from scratch to demonstrate
 a clear understanding of the mathematical foundations of CNNs. The obtained results
 were consistent with the theoretical calculations from Part 1, confirming the
 correctness of the implementation.
 
 The experiment also illustrated how parameters such as filter size, stride, and
 padding influence the output dimensions and data flow through the network. Batch
 normalization improved the stability of feature values, while the tanh activation
 function effectively constrained them within a specific range. The final softmax
 outputs and loss values indicated a properly functioning model. Overall, this
 exercise provided valuable insight into the internal mechanisms of CNNs and the
 importance of each processing step.


In [ ]:
import numpy as np

# Configure numpy printing for consistent formatting
np.set_printoptions(precision=3, suppress=True)

# Given data (from assignment)
X = np.array([
    [ 1, -1,  0,  2, -2],
    [ 2,  0, -1,  1,  1],
    [-2,  1,  2, -1,  3],
    [ 1, -2,  1,  2, -1],
    [ 3,  0, -1,  1,  1],
], dtype=float)

F1 = np.array([
    [ 1, 0, -2],
    [ 0, 1,  0],
    [-2, 0,  1],
], dtype=float)

F2 = np.array([
    [ 1, -1],
    [ 0,  1],
], dtype=float)

F3 = np.array([
    [0, 1, 0],
    [1,-2, 1],
    [0, 1, 0],
], dtype=float)

ALPHA = 0.1
EPS = 1e-2
GAMMA = 1.0
BETA = 0.0

w1 = np.array([ 0.5, -0.4,  0.3,  0.2, -0.1], dtype=float)
b1 = 0.1
w2 = np.array([-0.2,  0.6, -0.5,  0.4,  0.2], dtype=float)
b2 = -0.2

def pad2d(x, pad_h, pad_w):
    """Pad a 2D array with zeros on height and width dimensions.

    Args:
        x: 2D array of shape (H, W)
        pad_h: int, number of zeros to add to top and bottom
        pad_w: int, number of zeros to add to left and right

    Returns:
        Padded 2D array of shape (H + 2*pad_h, W + 2*pad_w)

    Notes:
        Uses constant zero padding via np.pad.
    """
    return np.pad(x, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant', constant_values=0.0)

## Primitives — implement here

In [ ]:
def conv2d(input, kernel, stride=1, padding='same'):
    """2D Convolution using cross-correlation (kernel NOT flipped, standard CNN practice).
    
    Args:
        input: 2D array of shape (H, W)
        kernel: 2D array of shape (kH, kW)
        stride: int, stride for convolution (must be >= 1)
        padding: str, 'same' or 'valid' padding mode
    
    Returns:
        2D array of shape (out_H, out_W) where:
        - out_H = (H + 2*pad_h - kH) // stride + 1  (for 'same' padding)
        - out_W = (W + 2*pad_w - kW) // stride + 1  (for 'same' padding)

    Notes:
        This implementation performs cross-correlation (no kernel flip). Raises
        ValueError for unsupported padding modes and AssertionError for invalid shapes/stride.
    """
    kH, kW = kernel.shape
    H, W = input.shape
    assert stride >= 1 and isinstance(stride, int), "stride must be an integer >= 1"
    if padding == 'same':
        pad_h = (kH - 1) // 2
        pad_w = (kW - 1) // 2
        x = pad2d(input, pad_h, pad_w)
    elif padding == 'valid':
        # ensure kernel fits inside input
        assert H >= kH and W >= kW, f"Input spatial dims ({H},{W}) must be >= kernel dims ({kH},{kW}) for 'valid' padding"
        x = input
    else:
        raise ValueError("padding must be 'same' or 'valid'")
    out_h = (x.shape[0] - kH) // stride + 1
    out_w = (x.shape[1] - kW) // stride + 1
    out = np.zeros((out_h, out_w), dtype=float)
    for i in range(out_h):
        for j in range(out_w):
            patch = x[i*stride:i*stride+kH, j*stride:j*stride+kW]
            out[i, j] = np.sum(patch * kernel)
    return out


def leaky_relu(x, alpha=0.1):
    """Leaky ReLU activation function.
    
    Args:
        x: 2D or 1D array of shape (...,)
        alpha: float, slope for negative values
    
    Returns:
        Array of same shape as input
    """
    return np.where(x >= 0, x, alpha * x)


def avg_pool2d(x, kernel_size=2, stride=1):
    """Average pooling over 2D feature maps.
    
    Args:
        x: 2D array of shape (H, W)
        kernel_size: int, pooling window size (kH, kW both = kernel_size)
        stride: int, stride for pooling (must be >= 1)
    
    Returns:
        2D array of shape (out_H, out_W) where:
        - out_H = (H - kernel_size) // stride + 1
        - out_W = (W - kernel_size) // stride + 1
    
    Raises:
        AssertionError if H < kernel_size or W < kernel_size
    """
    H, W = x.shape
    k = kernel_size
    assert isinstance(stride, int) and stride >= 1, "stride must be an integer >= 1"
    assert H >= k and W >= k, f"Input spatial dims ({H}, {W}) must be >= kernel_size ({k})"
    out_h = (H - k) // stride + 1
    out_w = (W - k) // stride + 1
    out = np.zeros((out_h, out_w), dtype=float)
    for i in range(out_h):
        for j in range(out_w):
            patch = x[i*stride:i*stride+k, j*stride:j*stride+k]
            out[i, j] = np.mean(patch)
    return out


def min_pool2d(x, kernel_size=2, stride=1):
    """Minimum pooling over 2D feature maps.
    
    Args:
        x: 2D array of shape (H, W)
        kernel_size: int, pooling window size (kH, kW both = kernel_size)
        stride: int, stride for pooling (must be >= 1)
    
    Returns:
        2D array of shape (out_H, out_W) where:
        - out_H = (H - kernel_size) // stride + 1
        - out_W = (W - kernel_size) // stride + 1
    
    Raises:
        AssertionError if H < kernel_size or W < kernel_size
    """
    H, W = x.shape
    k = kernel_size
    assert isinstance(stride, int) and stride >= 1, "stride must be an integer >= 1"
    assert H >= k and W >= k, f"Input spatial dims ({H}, {W}) must be >= kernel_size ({k})"
    out_h = (H - k) // stride + 1
    out_w = (W - k) // stride + 1
    out = np.zeros((out_h, out_w), dtype=float)
    for i in range(out_h):
        for j in range(out_w):
            patch = x[i*stride:i*stride+k, j*stride:j*stride+k]
            out[i, j] = np.min(patch)
    return out


def batch_norm(x, gamma=1.0, beta=0.0, eps=1e-2):
    """Batch normalization (per-feature statistics, unbiased=False).
    
    Args:
        x: 2D or 1D array of shape (...,)
        gamma: float, scale parameter (default 1.0)
        beta: float, shift parameter (default 0.0)
        eps: float, numerical stability constant (default 1e-2)
    
    Returns:
        Tuple of (normalized_output, mean, variance) where:
        - normalized_output: array of same shape as x
        - mean: float, E[x]
        - variance: float, Var[x] (computed with unbiased=False, matching np.var default)
    """
    mu = np.mean(x)
    var = np.var(x)  # unbiased=False by default (ddof=0)
    x_hat = (x - mu) / np.sqrt(var + eps)
    return gamma * x_hat + beta, mu, var


def tanh(x):
    """Hyperbolic tangent activation function.
    
    Args:
        x: array of any shape
    
    Returns:
        Array of same shape as input, values in [-1, 1]
    """
    return np.tanh(x)


def flatten(arrays):
    """Flatten and concatenate multiple 2D arrays into a 1D vector.
    
    Args:
        arrays: list/tuple of 2D arrays, or single 2D/1D array
    
    Returns:
        1D array of shape (total_elements,)
    """
    if isinstance(arrays, (list, tuple)):
        return np.concatenate([a.flatten() for a in arrays], axis=0)
    return arrays.flatten()


def fully_connected(x, W, b):
    """Fully connected (linear) layer.
    
    Contract: inputs -> scalar output
    - x: 1D array of shape (D,)
    - W: 1D array of shape (D,) - weight vector
    - b: scalar - bias term
    Returns: scalar W @ x + b
    
    Raises:
        AssertionError if dimensions mismatch.
    """
    x = np.asarray(x)
    W = np.asarray(W)
    assert x.ndim == 1, "x must be a 1D array"
    assert W.ndim == 1, "W must be a 1D array"
    assert W.shape[0] == x.shape[0], f"W length {W.shape[0]} must match x length {x.shape[0]}"
    return W @ x + b


def softmax(z):
    """Softmax function for probability distribution.
    
    Args:
        z: 1D array or list of logits of shape (C,)
    
    Returns:
        1D array of probabilities of shape (C,), sums to 1
    """
    z = np.array(z, dtype=float)
    z = z - np.max(z)
    e = np.exp(z)
    return e / np.sum(e)

## Forward Pass — run to verify each step

In [ ]:
# Step 1
s1 = conv2d(X, F1, stride=1, padding='same')
print('Step 1 shape:', s1.shape); print(s1)

# Step 2
s2 = leaky_relu(s1, alpha=ALPHA)
print('\nStep 2 shape:', s2.shape); print(s2)

# Step 3
s3 = avg_pool2d(s2, kernel_size=2, stride=1)
print('\nStep 3 shape:', s3.shape); print(s3)

# Step 4A (valid, stride=1)
A = conv2d(s3, F2, stride=1, padding='valid')
print('\nStep 4A shape:', A.shape); print(A)

# Step 4B (same, stride=2)
B = conv2d(s3, F3, stride=2, padding='same')
print('\nStep 4B shape:', B.shape); print(B)

# Step 5
A_pool = min_pool2d(A, kernel_size=2, stride=1)
B_pool = avg_pool2d(B, kernel_size=2, stride=1)
print('\nStep 5A MinPool shape:', A_pool.shape); print(A_pool)
print('\nStep 5B AvgPool shape:', B_pool.shape); print(B_pool)

# Step 6
A_bn, A_mu, A_var = batch_norm(A_pool, eps=EPS)
B_bn, B_mu, B_var = batch_norm(B_pool, eps=EPS)
print(f"\nStep 6A BN: mu={A_mu:.4f}, var={A_var:.4f}, shape={A_bn.shape}\n", A_bn)
print(f"\nStep 6B BN: mu={B_mu:.4f}, var={B_var:.4f}, shape={B_bn.shape}\n", B_bn)

# Step 7
A_t = tanh(A_bn)
B_t = tanh(B_bn)
print('\nStep 7A tanh:', A_t.shape); print(A_t)
print('\nStep 7B tanh:', B_t.shape); print(B_t)

# Step 8
x_flat = flatten([A_t, B_t])
print('\nStep 8 Flatten:', x_flat.shape); print(x_flat)

z1 = fully_connected(x_flat, w1, b1)
z2 = fully_connected(x_flat, w2, b2)
print('\nLogits:', z1, z2)

# Step 9
probs = softmax([z1, z2])
p1, p2 = probs
ce = -np.log(max(p1, 1e-12))
mse = 0.5 * ((1 - p1)**2 + (0 - p2)**2)
print(f"\nSoftmax: p1={p1:.6f}, p2={p2:.6f}")
print(f"Cross-Entropy (Class 0): {ce:.6f}\nMSE: {mse:.6f}")


In [ ]:
# Sanity-check: compare current Part II outputs to Part I arrays (if available)
# The cell will look for common Part I variable name variants and compare with np.allclose.

vars_to_check = [
    's1','s2','s3','A','B','A_pool','B_pool','A_bn','B_bn','A_t','B_t','x_flat','z1','z2','probs'
]

found_any = False
all_ok = True

for name in vars_to_check:
    # Candidate Part I names to look for
    candidates = [
        f'part1_{name}',
        f'Part1_{name}',
        f'P1_{name}',
        f'{name}_part1',
        f'{name}_P1'
    ]
    if name not in globals():
        # current variable not present (maybe cell not executed yet)
        print(f"Skipping {name}: current variable not defined")
        continue
    cur = globals()[name]
    matched = False
    for cand in candidates:
        if cand in globals():
            found_any = True
            ref = globals()[cand]
            try:
                ok = np.allclose(cur, ref, atol=1e-6)
            except Exception:
                # For scalars or incompatible shapes compare as float
                try:
                    ok = float(cur) == float(ref)
                except Exception:
                    ok = False
            print(f"Compare {name} vs {cand}:", 'OK' if ok else 'DIFFER')
            all_ok = all_ok and ok
            matched = True
    if not matched:
        print(f"No Part I match found for {name} (tried {candidates})")

if not found_any:
    print('No Part I reference variables found in the notebook namespace. Define them (e.g. part1_s1) to run comparison.')
else:
    if all_ok:
        print('OK')
    else:
        print('Some comparisons failed. Review printed differences.')